In [0]:
from pyspark.sql.functions import col, trim, to_date, sha2, concat_ws

bronze_employee_df = spark.sql("""
SELECT *
FROM hr_poc.silver.employee
""")
display(bronze_employee_df.limit(5))

In [0]:
employee_new_df = (
    bronze_employee_df
    .withColumn("employee_id", trim(col("employee_id")))
    .withColumn("employee_name", trim(col("employee_name")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("date_of_birth", to_date(col("date_of_birth"), "M/d/yyyy"))
    .withColumn("joining_date", to_date(col("joining_date"), "M/d/yyyy"))
    .withColumn("department_id", trim(col("department_id")))
    .withColumn("Designation_id", trim(col("Designation_id")))
    .withColumn("location", trim(col("location")))
    .withColumn("employment_type", trim(col("employment_type")))
    .withColumn("manager_id", trim(col("manager_id").cast("string")))
    .withColumn("status", trim(col("status")))
    .dropDuplicates(["employee_id"])
)

employee_new_df.createOrReplaceTempView("employee_new_stage")

display(employee_new_df.limit(5))

In [0]:
employee_new_df.createOrReplaceTempView("employee_new_clean")

display(spark.sql("""
WITH duplicate_employee_ids AS (
    SELECT employee_id
    FROM employee_new_clean
    GROUP BY employee_id
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT employee_id) AS distinct_employee_ids,
    SUM(CASE WHEN employee_id IS NULL OR employee_id = '' THEN 1 ELSE 0 END) AS null_employee_ids,
    (SELECT COUNT(*) FROM duplicate_employee_ids) AS duplicate_employee_id_groups
FROM employee_new_clean
"""))


In [0]:
employee_new_scd_df = employee_new_df.withColumn(
    "change_hash",
    sha2(
        concat_ws(
            "||",
            col("department_id").cast("string"),
            col("Designation_id").cast("string"),
            col("location"),
            col("employment_type"),
            col("manager_id").cast("string"),
            col("status")
        ),
        256
    )
)

employee_new_scd_df.createOrReplaceTempView("employee_new_scd")

display(employee_new_scd_df.select(
    "employee_id",
    "department_id",
    "Designation_id",
    "location",
    "change_hash"
).limit(5))

In [0]:
gold_current_df = (
    spark.table("hr_poc.gold.fact_employee")
    .filter(col("is_current") == "Y")
)

gold_current_df.createOrReplaceTempView("gold_current_employee")

display(spark.sql("""
SELECT
    COUNT(*) AS current_gold_rows,
    COUNT(DISTINCT employee_id) AS current_gold_distinct_employees
FROM gold_current_employee
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW employee_scd2_candidates AS
SELECT
    s.*,
    CASE
        WHEN g.employee_id IS NULL THEN 'INSERT'
        WHEN s.change_hash <> g.change_hash THEN 'UPDATE_INSERT'
    END AS scd_action
FROM employee_new_scd s
LEFT JOIN gold_current_employee g
    ON s.employee_id = g.employee_id
WHERE g.employee_id IS NULL
   OR s.change_hash <> g.change_hash
""")

display(spark.sql("""
SELECT
    scd_action,
    COUNT(*) AS employee_count
FROM employee_scd2_candidates
GROUP BY scd_action
ORDER BY scd_action
"""))

In [0]:
display(spark.sql("""
SELECT
    employee_id,
    scd_action,
    department_id,
    Designation_id,
    location
FROM employee_scd2_candidates
ORDER BY employee_id
LIMIT 20
"""))


In [0]:
spark.sql("""
MERGE INTO hr_poc.gold.fact_employee AS g
USING (
    SELECT employee_id
    FROM employee_scd2_candidates
    WHERE scd_action = 'UPDATE_INSERT'
) AS s
ON g.employee_id = s.employee_id
AND g.is_current = 'Y'
WHEN MATCHED THEN
UPDATE SET
    g.effective_end_date = current_date(),
    g.is_current = 'N'
""")

display(spark.sql("""
SELECT
    COUNT(*) AS historical_rows
FROM hr_poc.gold.fact_employee
WHERE is_current = 'N'
"""))

In [0]:
spark.sql("""
INSERT INTO hr_poc.gold.fact_employee (
    employee_id,
    employee_name,
    gender,
    date_of_birth,
    joining_date,
    department_id,
    Designation_id,
    location,
    employment_type,
    manager_id,
    status,
    change_hash,
    employee_skey,
    effective_start_date,
    effective_end_date,
    is_current
)
WITH max_key AS (
    SELECT COALESCE(MAX(employee_skey), 0) AS max_employee_skey
    FROM hr_poc.gold.fact_employee
)
SELECT
    c.employee_id,
    c.employee_name,
    c.gender,
    c.date_of_birth,
    c.joining_date,
    c.department_id,
    c.Designation_id,
    c.location,
    c.employment_type,
    c.manager_id,
    c.status,
    c.change_hash,
    mk.max_employee_skey + ROW_NUMBER() OVER (ORDER BY c.employee_id, c.scd_action) AS employee_skey,
    current_date() AS effective_start_date,
    CAST(NULL AS DATE) AS effective_end_date,
    'Y' AS is_current
FROM employee_scd2_candidates c
CROSS JOIN max_key mk
""")

In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT employee_id) AS distinct_employees,
    SUM(CASE WHEN is_current = 'Y' THEN 1 ELSE 0 END) AS current_rows,
    SUM(CASE WHEN is_current = 'N' THEN 1 ELSE 0 END) AS historical_rows
FROM hr_poc.gold.fact_employee
"""))

In [0]:
display(spark.sql("""
SELECT
    employee_id,
    department_id,
    Designation_id,
    location,
    effective_start_date,
    effective_end_date,
    is_current
FROM hr_poc.gold.fact_employee
WHERE employee_id IN ('E1001', 'E1003', 'E1005', 'E1011', 'E1201')
ORDER BY employee_id, effective_start_date, is_current
"""))